In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
)

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


class State(TypedDict):
    prompt: str
    response: str
    summary: str


def node_1(state: State):
    user_input = state["prompt"]

    response = llm.invoke(
        f"User said: {user_input}. Respond with output, no explanation."
    )

    return {"response": response.content}


def node_2(state: State):
    response = state["response"]

    summary = llm.invoke(
        f"Summarize this output: {response}"
    )

    return {"summary": summary.content}


builder = StateGraph(State)

builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)

builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()
graph

In [11]:
# Invloking graph


graph.invoke({"prompt": "capital of India"})




{'prompt': 'capital of India',
 'response': 'New Delhi',
 'summary': 'New Delhi is the capital of India, a major city known for its historical significance, political importance, and cultural landmarks. It serves as the seat of the Government of India, housing key institutions like the Parliament and Supreme Court. The city blends modern infrastructure with colonial-era architecture and is a hub for politics, commerce, and culture in the country. With a population of over 25 million, it is one of the most populous cities globally.'}